# ZAIC 2026 LifeMate — LLM CUỐI (public 52.10)

Tái hiện `ft_v11_all_adapter`: Qwen3-4B-Instruct-2507 + LoRA, train trên **toàn bộ 1545 mẫu train** + 440 spec `gen_*` (aug 1)
+ 320 `llm_*` + 285 `claude_*` (aug 2, seed 1), drop-empty. Public đo với ASR `asr_v8_large_all`.
Cấu hình đã đặt sẵn — **không sửa gì, chỉ Run all trên A100** (~2.5 giờ train, ~3.5 giờ tổng).

Kiểm tra: cell 1 in `TAG: v11_all | ASR_PROD: .../asr_v8_large_all`; cell 2 in `OK:`; GPU A100; `synth_llm: 605`;
`sft_real ... from 1545 samples`; SFT = 5455. Val trong lần chạy này nằm trong tập train nên điểm val không có ý nghĩa.
Cần trên Drive (`MyDrive/ZAIC/`): `lifemate_baseline.zip` (zip của repo), `asr_v8_large_all/`.


In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BATCH      = 32
LLM_EPOCHS = 1
TRAIN_ALL  = True     # False: train trên train_fit, đo được val. True: cả 1545 mẫu — chỉ cho lần train CUỐI
USE_ASR    = False    # giữ False (bài học v5)
DROP_EMPTY = True     # bỏ mẫu synthetic có đáp án [] (bài học v6)
GEN_AUG    = 1        # số bản augment cho spec sinh bằng code (như v9)
LLM_AUG    = 2        # số bản augment cho spec viết tay (như v9)
USE_CLAUDE = True     # True: thêm 285 spec synth_specs_claude (v11). False: đúng dữ liệu v9
USE_C12    = False    # True: thêm 131 spec synth_specs_c12 (v12)
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ASR_PROD   = DRIVE_DIR + "/asr_v8_large_all"   # ASR cuối (Whisper large-v3, train trên toàn bộ clip)
TAG        = ("v12" if USE_C12 else "v11" if USE_CLAUDE else "v9") + ("_all" if TRAIN_ALL else "")
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", f"/content/results_{TAG}"
LLM_OUT    = f"/content/ft_{TAG}"
GEN_GLOBS  = [WORK + "/synth_specs/gen_*.jsonl"]
CLAUDE_DIR = WORK + "/synth_specs_claude"      # nằm sẵn trong lifemate_baseline.zip
C12_DIR    = WORK + "/synth_specs_c12"
print("TAG:", TAG, "| ASR_PROD:", ASR_PROD)   # phải là: TAG: v11_all | ASR_PROD: .../asr_v8_large_all


In [ ]:
# ==== KIỂM TRA ĐÚNG FILE (chạy đầu tiên) ====
NOTEBOOK = "final_llm"
from google.colab import drive; drive.mount("/content/drive")
import os, zipfile
_z = os.path.join(DRIVE_DIR, "lifemate_baseline.zip")
assert os.path.exists(_z), "Drive không có MyDrive/ZAIC/lifemate_baseline.zip"
_v = zipfile.ZipFile(_z).read("VERSION").decode().strip() if "VERSION" in zipfile.ZipFile(_z).namelist() else "(không có VERSION — zip cũ)"
assert _v == "2026-11-13-2300", f"lifemate_baseline.zip trên Drive là bản {_v}, cần bản 2026-11-13-2300 → ghi đè bằng bản mới nhất (zip của repo)"
_need = ["asr_v8_large_all/config.json"]
_miss = [p for p in _need if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not _miss, "Thiếu trên Drive: " + ", ".join(_miss)
print(f"OK: notebook {NOTEBOOK} | zip {_v} | đủ: {_need}")


In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
gen_specs = sorted(p for g in GEN_GLOBS for p in glob.glob(g))
# spec viết tay của v9: đúng 320 mẫu đã cho public 47.17 (synth_specs/llm_*.jsonl trong zip)
v9 = sorted(glob.glob(WORK + "/synth_specs/llm_*.jsonl"))
cl = sorted(glob.glob(CLAUDE_DIR + "/claude_*.jsonl")) if USE_CLAUDE else []
c12 = sorted(glob.glob(C12_DIR + "/c12_*.jsonl")) if USE_C12 else []
llm_specs = v9 + cl + c12
n_v9_lines = sum(1 for p in v9 for _ in open(p))
assert gen_specs and n_v9_lines == 320, f"spec v9 phải đúng 320 dòng, đang {n_v9_lines} — lifemate_baseline.zip trên Drive chưa phải bản mới nhất"
assert not USE_C12 or len(c12) == 4, "thiếu synth_specs_c12 — lifemate_baseline.zip trên Drive chưa phải bản mới nhất"
assert not USE_CLAUDE or len(cl) == 11, "thiếu synth_specs_claude — lifemate_baseline.zip trên Drive chưa phải bản mới nhất"
assert not glob.glob(WORK + "/synth_specs_v10*"), "zip còn dữ liệu v10 (bị loại) — dùng bản zip mới nhất"
print("gen:", len(gen_specs), "| llm:", len(llm_specs), [os.path.basename(p) for p in llm_specs])
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
IDS  = "" if TRAIN_ALL else "--ids splits/train_fit.txt"
ASRF = "--asr asr_train_oof.json" if USE_ASR else ""
if USE_ASR:
    shutil.copy(os.path.join(DRIVE_DIR, "asr_train_oof.json"), WORK + "/asr_train_oof.json")
print("DATA:", DATA)

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-800:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. Dựng và kiểm tra dữ liệu synthetic bằng simulator của BTC

In [ ]:
%cd {WORK}
!python -m lifemate.synth_build --root {DATA} --tools-from public_test --specs {' '.join(gen_specs)} --split synth_gen 2>&1 | tee {RES}/synth_build_gen.log | tail -5
!python -m lifemate.synth_build --root {DATA} --tools-from public_test --specs {' '.join(llm_specs)} --split synth_llm 2>&1 | tee {RES}/synth_build_llm.log | tail -5


In [ ]:
import collections as C
tag = lambda r: "oos" if not r["label"]["tool_calls"] else ("ask" if any(c["name"].startswith("sys.") for c in r["label"]["tool_calls"]) else "act")
stats, n_synth = [], 0
for sp in ["synth_gen", "synth_llm"]:
    rows = [json.loads(l) for l in open(f"{DATA}/{sp}/{sp}.jsonl", encoding="utf-8")]
    apps = C.Counter(a for r in rows for a in r["context"]["installed_apps"])
    line = f"{sp}: {len(rows)} mẫu | {dict(C.Counter(tag(r) for r in rows))} | {apps.most_common(12)}"
    print(line); stats.append(line); n_synth += len(rows)
n_llm = sum(1 for _ in open(f"{DATA}/synth_llm/synth_llm.jsonl"))
want = 320 + (285 if USE_CLAUDE else 0) + (131 if USE_C12 else 0)
print("synth_llm:", n_llm, "mẫu, phải đúng", want)
assert n_llm == want, "số mẫu viết tay không khớp — xem synth_build_llm.log"
open(f"{RES}/synth_stats.txt", "w").write("\n".join(stats) + "\n")


## 2. Dữ liệu SFT (công thức v4: bản gốc + bản đổi tên app)

In [ ]:
%cd {WORK}
DE = "--drop-empty" if DROP_EMPTY else ""
!python -m lifemate.sft_data --root {DATA} {IDS} --aug-copies 1 {ASRF} --out sft_real.jsonl
!python -m lifemate.sft_data --root {DATA} --split synth_gen --aug-copies {GEN_AUG} {DE} --out sft_gen.jsonl
!python -m lifemate.sft_data --root {DATA} --split synth_llm --aug-copies {LLM_AUG} {DE} --seed 1 --out sft_llm.jsonl
with open("sft_all.jsonl", "w", encoding="utf-8") as out:
    for p in ["sft_real.jsonl", "sft_gen.jsonl", "sft_llm.jsonl"]:
        out.writelines(open(p, encoding="utf-8"))
n = sum(1 for _ in open("sft_all.jsonl")); print("tổng ví dụ SFT:", n, "(v9: 3562; v11 = 4279; v12 = 4672)")


## 3. Train (full gradient như v4 — **không** dùng `--detach-prefix`). A100 khoảng 2–3 giờ

In [ ]:
%cd {WORK}
!python -m lifemate.train_lora --data sft_all.jsonl --model {BASE_MODEL} --out {LLM_OUT} --epochs {LLM_EPOCHS} 2>&1 | grep -vi warn
assert os.path.exists(f"{LLM_OUT}/merged/config.json"), "train lỗi: xem log phía trên"
shutil.copy(f"{LLM_OUT}/train_log.txt", f"{RES}/train_log.txt")
shutil.copytree(f"{LLM_OUT}/adapter", os.path.join(DRIVE_DIR, f"ft_{TAG}_adapter"), dirs_exist_ok=True)

## 4. Đánh giá trên val với ASR thật (mốc cần vượt: v9 39.52 / 58.31)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {ASR_PROD} --dtype {DTYPE} --out asr_val.json 2>&1 | grep -E "^model=|^WER"
sc = {}
for S in ["val_unseen", "val_seen"]:
    !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --transcripts asr_val.json --out runs/{TAG}_{S} 2>&1 | tail -1
    !python scripts/score.py --root {DATA} --split train --pred runs/{TAG}_{S}/submission.jsonl --by-tag > {RES}/score_{TAG}_{S}.txt
    !python -m lifemate.report --root {DATA} --split train --pred runs/{TAG}_{S}/submission.jsonl --show 6 > {RES}/report_{TAG}_{S}.txt
    sc[S] = float(open(f"{RES}/score_{TAG}_{S}.txt").readline().split()[-1])
    !head -9 {RES}/report_{TAG}_{S}.txt
print(TAG, sc, "| v9: {'val_unseen': 39.52, 'val_seen': 58.31} | v10 (loại): 39.70 / 55.87")
if TRAIN_ALL:
    print("CHÚ Ý: TRAIN_ALL=True nên val đã nằm trong tập train — điểm này cao giả tạo, chỉ nhìn điểm public")
json.dump(sc, open(f"{RES}/e2e_{TAG}.json", "w"))

## 5. Public → `submission_public_{TAG}.jsonl` (mốc cần vượt: **47.17** của v9)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split public_test --model {ASR_PROD} --dtype {DTYPE} --out asr_public.json 2>&1 | tail -1
!python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --transcripts asr_public.json --out runs/public_{TAG} 2>&1 | tail -1
shutil.copy(f"runs/public_{TAG}/submission.jsonl", f"{RES}/submission_public_{TAG}.jsonl")
shutil.copy(f"runs/public_{TAG}/submission.jsonl", os.path.join(DRIVE_DIR, f"submission_public_{TAG}.jsonl"))
shutil.copy(f"runs/public_{TAG}/traces.jsonl", f"{RES}/traces_public_{TAG}.jsonl")
shutil.copy("asr_public.json", f"{RES}/asr_public.json")
sub = [json.loads(l) for l in open(f"runs/public_{TAG}/submission.jsonl")]
ask = sum(1 for x in sub if any(c["name"].startswith("sys.") for c in x["tool_calls"]))
empty = sum(1 for x in sub if not x["tool_calls"])
print(f"dòng {len(sub)} | ask_missing_info {ask} (v9: 54, v4: 92) | rỗng {empty} (v9: 85, v4: 64)")

In [ ]:
%cd {DATA}/public_test
!python -m zaic_tools --split public_test --root .. --validate {RES}/submission_public_{TAG}.jsonl 2>&1 | tail -3 | tee {RES}/validate_public_{TAG}.txt

## 5b. Val với transcript CHUẨN (đo ASR làm mất bao nhiêu điểm)


In [ ]:
%cd {WORK}
gold = {}
for S in ["val_unseen", "val_seen"]:
    !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --out runs/{TAG}_gold_{S} 2>&1 | tail -1
    !python scripts/score.py --root {DATA} --split train --pred runs/{TAG}_gold_{S}/submission.jsonl --by-tag > {RES}/score_{TAG}_gold_{S}.txt
    gold[S] = float(open(f"{RES}/score_{TAG}_gold_{S}.txt").readline().split()[-1])
print("transcript chuẩn:", gold, "| ASR:", sc, "| ASR làm mất:", {k: round(gold[k] - sc[k], 2) for k in gold})
json.dump({"gold": gold, "asr": sc}, open(f"{RES}/asr_gap_{TAG}.json", "w"))


## 6. Đóng gói

In [ ]:
open(f"{RES}/config.txt", "w").write(f"BASE={BASE_MODEL}\nTRAIN_ALL={TRAIN_ALL}\nUSE_ASR={USE_ASR}\nDROP_EMPTY={DROP_EMPTY}\nGEN_AUG={GEN_AUG}\nLLM_AUG={LLM_AUG}\nUSE_CLAUDE={USE_CLAUDE}\nUSE_C12={USE_C12}\nSFT={n}\nsynth={n_synth}\nGPU={GPU_NAME}\n")
z = shutil.make_archive(f"/content/results_{TAG}", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, f"results_{TAG}.zip")); print(f"saved results_{TAG}.zip")
from google.colab import files
files.download(z); files.download(f"{RES}/submission_public_{TAG}.jsonl")